## Load segments from Slicer

In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, tetra_mesh_from_stl, tetra_shell_from_two_surfaces, set_the_offset, export_centerline


pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

# offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")


in_folder = "segment_final/"    
out_folder = "spline_experiment/"

folder = "spline_meshes/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)




filename = "Segmentation_1.seg.nrrd"
export_name = base + folder + filename[:-8]

eyeball_id = 1
muslce_ids = [2,3,4,5]
lense_id = 7

background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)


print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




(110, 150, 155)
pixel spacing =  [0.5 0.5 0.5]
id =  7
dimensions =  (110, 150, 155)
dimensions =  (110, 150, 155)
id =  7


In [2]:
muscles = []

for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))


            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:

                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

                mask_eye = mask

            if id in muslce_ids:
                print("Muscle")

                muscles.append(mask)
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )

            if id == lense_id:
                mask = ndimage.median_filter(mask, size=3)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )


id =  1
65767
Problem edges: 0
Outer and inner surfaces written.
id =  2
8436
Muscle
Problem edges: 4
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._surf_id_2.stl
id =  3
4587
Muscle
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._surf_id_3.stl
id =  4
7222
Muscle
Problem edges: 1
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._surf_id_4.stl
id =  5
6801
Muscle
Problem edges: 6
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._surf_id_5.stl
id =  6
6465
id =  7
1372
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._surf_id_7.stl


In [3]:
mesh = trimesh.load(export_name +"_surf_id_"+str(lense_id)+".stl")
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


def eyeball_axis(t):
    return lense_center + t * axis_direction


axis_points = [eyeball_axis(-20.0), eyeball_axis(20.0)]

export_centerline(axis_points, 1.0 , base + folder + "lense_axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point =  eyeball_axis(-2.0)/pixel_spacing[0]

print("target_point = ", target_point)

lense_center =  [34.8451185  63.80259441 24.46803607]
target_point =  [ 69.51639054 123.70304501  49.79807623]


In [4]:
def fit_target_constrained_spline(binned_centers, target_point, smoothing_factor=1.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    print("target_point = ", target_point)
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)

    print("     constrained_points = ", constrained_points)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=3)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-0.1, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    


def fit_analytical_centerline(mask, target_point, ignored_nodes, n_bins=20, extension_factor=0.2):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component (the "length" of the object)
        projection = pca.fit_transform(points)
        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Instead of fitting to 10,000 points, we fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        # # Smoothing 's' can be lower here because the binning already cleaned the noise
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        # # Evaluating from 0.0 to 1+extension extends the centerline
        smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0, 100), tck)).T

        smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point)
        
        return smooth_centerline, smooth_centerline_extended

In [ ]:



def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.9 else numpy.array([0, 1, 0])
    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2

def reconstruct_variable_volume(centerline, centerline_extended, init_mesh_surf, original_shape, extension_factor=0.2):
    # 1. Setup
    # mask_points = numpy.argwhere(original_mask == 1)
    # tree_points = cKDTree(mask_points)

    mesh_points = numpy.array(init_mesh_surf.vertices/pixel_spacing[0])   
    mesh_tree = cKDTree(mesh_points)
    
    n_angles = 16
    angles = numpy.linspace(0, 2*numpy.pi, n_angles, endpoint=False)
    radial_profiles = [] 
    v1_list = []
    v2_list = []

    # 1.5 PRECOMPUTE the v1, v2
    for i, p in enumerate(centerline_extended):
        direction = centerline_extended[min(i+1, len(centerline_extended)-1)] - centerline_extended[max(i-1, 0)]
        direction = direction/numpy.linalg.norm(direction)
        v1, v2 = get_local_frame(direction)

        # Get the frame and SAVE IT
        v1_list.append(v1)
        v2_list.append(v2)
        
    v1_list = numpy.array(v1_list)
    v2_list = numpy.array(v2_list)

    # 2. SAMPLING: Map the original shape to 8 radial "ribs"
    for i, p in enumerate(centerline):
        direction = centerline[min(i+1, len(centerline)-1)] - centerline[max(i-1, 0)]
        direction = direction/numpy.linalg.norm(direction)
        v1, v2 = get_local_frame(direction)
        
        # Query the mesh surface 
        # idx = mesh_tree.query_ball_point(p, r=45.0) 

        # 1. Start with a slightly smaller ball to get candidates
        idx = mesh_tree.query_ball_point(p, r=45.0)
        local_pts = mesh_points[idx] - p

        # 2. Project points onto the spine direction (Normal to the plane)
        # This tells us how far 'above' or 'below' the plane each point is
        dist_from_plane = numpy.dot(local_pts, direction)

        # 3. Keep only points within a thin 'slab' (e.g., 2 units thick)
        thickness = 10.0
        mask_slab = numpy.abs(dist_from_plane) < (thickness / 2.0)
        local_pts = local_pts[mask_slab]
                
        if len(idx) > 5:
            local_pts = mesh_points[idx] - p
            x_p = numpy.dot(local_pts, v1)
            y_p = numpy.dot(local_pts, v2)
            
            pt_angles = numpy.arctan2(y_p, x_p) % (2 * numpy.pi)
            pt_dists = numpy.sqrt(x_p**2 + y_p**2)
            
            current_profile = []
            for a in angles:
                angular_diff = numpy.abs(pt_angles - a)
                angular_diff = numpy.minimum(angular_diff, 2*numpy.pi - angular_diff)
                
                # Mesh vertices are already "on the surface", 
                mask_slice = angular_diff < (numpy.pi / n_angles)

                if numpy.any(mask_slice):
                    current_profile.append(numpy.median(pt_dists[mask_slice]))
                else:
                    current_profile.append(numpy.nan)
            radial_profiles.append(current_profile)
        else:
            radial_profiles.append([numpy.nan]*n_angles)

    # 3. SMOOTHING: Use splines to clean the "ribs" and extend them
    radial_profiles = numpy.array(radial_profiles)
    
    u = numpy.linspace(0, 1, len(radial_profiles))
    u_full = numpy.linspace(0, 1, len(centerline_extended))
    smooth_profiles = numpy.zeros((len(centerline_extended), n_angles))

    for a in range(n_angles):
        column = radial_profiles[:, a]
        valid = ~numpy.isnan(column)
        
        if numpy.sum(valid) > 4:
            # k=3 (Cubic) is generally more stable for extrapolation than k=4
            rib_spline = UnivariateSpline(u[valid], column[valid], k=3, s=20)
            
            # Evaluate for the full extended centerline
            extrapolated_radii = rib_spline(u_full)
            
            # Safety: Ensure the radius never goes negative and 
            # doesn't grow unrealistically beyond the last known 'truth'
            last_valid_radius = column[valid][-1]
            first_valid_radius = column[valid][0]
            
            # Identify where we are extrapolating (outside the 'valid' u range)
            u_start, u_end = u[valid][0], u[valid][-1]
            
            # Option A: Constant Extension (Very stable)
            extrapolated_radii[u_full > u_end] = last_valid_radius
            extrapolated_radii[u_full < u_start] = first_valid_radius
            
            smooth_profiles[:, a] = extrapolated_radii
        else:
            smooth_profiles[:, a] = 1.0


    # 4. RECONSTRUCTION: Build the 3D volume
    new_mask = numpy.zeros(original_shape, dtype=numpy.uint8)
    max_r = smooth_profiles.max()

    
    # Bounding box logic
    z_min, y_min, x_min = numpy.floor(centerline_extended.min(axis=0) - max_r - 1).astype(int).clip(0)
    z_max, y_max, x_max = numpy.ceil(centerline_extended.max(axis=0) + max_r + 1).astype(int)
    z_max = min(z_max, original_shape[0]); y_max = min(y_max, original_shape[1]); x_max = min(x_max, original_shape[2])

    z_grid, y_grid, x_grid = numpy.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
    grid_coords = numpy.stack(numpy.meshgrid(z_grid.ravel(), y_grid.ravel(), x_grid.ravel(), indexing='ij'), axis=-1)

    tree_spine = cKDTree(centerline_extended)
    dist_to_spine, idx_on_spine = tree_spine.query(grid_coords)

    # Orientation logic
    diffs = grid_coords - centerline_extended[idx_on_spine]
    v1_local = v1_list[idx_on_spine]
    v2_local = v2_list[idx_on_spine]

    # Project into local 2D space
    dots_v1 = numpy.einsum('ij,ij->i', diffs.reshape(-1, 3), v1_local.reshape(-1, 3))
    dots_v2 = numpy.einsum('ij,ij->i', diffs.reshape(-1, 3), v2_local.reshape(-1, 3))

    voxel_angles = numpy.arctan2(dots_v2, dots_v1) % (2 * numpy.pi)
    
    # Linear interpolation between the smoothed radial ribs
    angle_idx = (voxel_angles / (2 * numpy.pi) * n_angles)
    idx_low = numpy.floor(angle_idx).astype(int) % n_angles
    idx_high = (idx_low + 1) % n_angles
    weight = angle_idx - numpy.floor(angle_idx)

    r_low = smooth_profiles[idx_on_spine.flatten(), idx_low]
    r_high = smooth_profiles[idx_on_spine.flatten(), idx_high]
    local_threshold_radii = (1 - weight) * r_low + weight * r_high

    # Create mask: True if voxel is within the analytical surface
    local_mask = (dist_to_spine.flatten() <= local_threshold_radii).astype(numpy.uint8)
    new_mask[z_min:z_max, y_min:y_max, x_min:x_max] = local_mask.reshape(z_grid.size, y_grid.size, x_grid.size)

    return new_mask

In [6]:
print(muscles[0].shape)

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    init_mesh = trimesh.load(export_name +"_surf_id_"+str(muslce_ids[muscle_id])+".stl")

    ignored_nodes = 0.01


    # --- Usage ---
    # Replace the skeleton logic with this:
    centerline, centerline_ext = fit_analytical_centerline(mask, target_point, ignored_nodes, n_bins=10, extension_factor=0.25)




    export_centerline(centerline_ext, pixel_spacing[0], base + folder + "extended_centerline_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing[0], base + folder + "centerline_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])


    new_mask = reconstruct_variable_volume(centerline, centerline_ext, init_mesh, data.shape, extension_factor=0.2)
    new_mask[mask_eye==1]=0

    mesh_surf = getSurfaceMesh(new_mask, export_name +"_NEW_surf_id_"+str(muslce_ids[muscle_id])+".stl", pixel_spacing[0], False )


(110, 150, 155)
target_point =  [ 69.51639054 123.70304501  49.79807623]
     constrained_points =  [[ 45.84009009  46.88738739  50.60585586]
 [ 52.13583441  51.2690815   50.78266494]
 [ 57.70116156  56.3093981   50.35480465]
 [ 62.92225859  61.71849427  50.28723404]
 [ 68.58271787  66.40915805  50.42836041]
 [ 74.3264      71.4888      50.8656    ]
 [ 78.48235294  77.69607843  50.53529412]
 [ 82.71392405  84.18860759  50.04936709]
 [ 86.63090129  90.21030043  49.88841202]
 [ 69.51639054 123.70304501  49.79807623]]
Problem edges: 9
written file:  /Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/spline_meshes/Segmentation_1._NEW_surf_id_2.stl
target_point =  [ 69.51639054 123.70304501  49.79807623]
     constrained_points =  [[ 41.9787234   59.03829787  64.84255319]
 [ 46.08130081  62.84146341  67.42276423]
 [ 48.42972536  67.63974152  70.0371567 ]
 [ 51.06528662  72.55095541  72.3933121 ]
 [ 53.27083333  77.89423077  74.32211538]
 [ 56.34129693  82.9